# Suite BND — Typed bindings

`Bindings` binds a program's own class to a reference object schema: the class gives `read(instance)` and
`make(state)`, between its fields and the schema's properties and entries, and the binding does the rest: `accept`,
a builder that implements `Visitors.OfObject` over a state, and a registry for `Plain.FromPlain`. A binding may fix a
property's value, make properties exclusive, and leave an adjacency implied by its other ends.

In [ ]:
from dataclasses import dataclass, field

from mbse.Schemas.Framework import Bindings as Bd, Comparison as C, JSON, Proxies, Schemas as S, Validators, YAML
from support import raises, text

store = Proxies.OfStore()

Note = S.OfObject.Builder().properties(text("text")).create()
Link = S.OfRelation.Builder().links("source", "target").properties(text("weight", int),
                                                                   lambda p: p.name("note").of(Note)).create()
Shape = S.OfObject.Builder().properties(text("w", int)).create()
Node = S.OfObject.Builder().ref().properties(
    text("tag"), text("label"), text("i", int), text("f", float), lambda p: p.name("shape").of(Shape),
    lambda p: p.name("tags").of(lambda t: t.as_indexed(lambda i: i.of(lambda t: t.as_native(str))))).relations(
    lambda a: a.name("out").of(Link).me("source"), lambda a: a.name("into").of(Link).me("target")).create()
store.register("Node", Node)
store.register("Link", Link)


@dataclass(eq=False)
class Point:
    """A class of the program's own, bound to `Node`: its number is `i` or `f`, its shape and tags are plain data, and
    `out` holds (target, weight, note) triples."""

    label: str | None = None
    number: int | float | None = None
    shape: dict | None = None
    tags: list | None = None
    out: list = field(default_factory=list)

    def identity(self):
        return id(self)

    def schema_name(self):
        return "Node"

    def owner(self):
        return None

    def accept(self, visitor):
        Bd.accept(BINDING, self, visitor)


def read(point):
    values = {"label": point.label, "shape": point.shape, "tags": point.tags}
    if point.number is not None:
        values["i" if type(point.number) is int else "f"] = point.number
    return Bd.State(values, {"out": [Bd.Entry({"target": t}, {"weight": w, "note": n}) for t, w, n in point.out]})


def make(state):
    v = state.values
    return Point(v.get("label"), v.get("i", v.get("f")), v.get("shape"), v.get("tags"),
                 [(e.links.get("target"), e.properties.get("weight"), e.properties.get("note"))
                  for e in state.entries.get("out", [])])


def assign(point, state):
    point.__dict__.update(make(state).__dict__)
    return point


BINDING = Bd.Binding(Node, read, make, assign, fixed={"tag": "node"}, exclusive=[("i", "f")], implied=["into"])
B = Bd.OfStore({"Node": (Node, lambda instance=None: Bd.Builder(BINDING, instance))}, {"Link": Link})
native = lambda value: lambda p: p.value(lambda a: a.as_native(lambda n: n.set(value)))  # noqa: E731

## BND-01 · A bound class writes itself through its binding, and is rebuilt from snapshots with the registry

In [ ]:
b = Point("b", 1.5, tags=["x"])
a = Point("a", 5, {"w": 3}, ["p", "q"], [(b, 7, {"text": "hi"}), (b, None, None)])
text_ = JSON.ToJSON(store).Reachable(Node, a)
assert text_ == (
    '{"root": "s0", "objects": {"s0": {"tag": "node", "label": "a", "i": 5, "shape": {"w": 3}, "tags": ["p", "q"], '
    '"out": [{"target": {"$ref": "s1", "$schema": "Node"}, "weight": 7, "note": {"text": "hi"}}, '
    '{"target": {"$ref": "s1", "$schema": "Node"}}]}, "s1": {"tag": "node", "label": "b", "f": 1.5, "tags": ["x"]}}}')
back = JSON.FromJSON(B).Reachable(Node, text_)
assert isinstance(back, Point) and back.number == 5 and back.shape == {"w": 3} and back.tags == ["p", "q"]
assert back.out[0][0].number == 1.5 and back.out[0][1:] == (7, {"text": "hi"}) and back.out[1][1:] == (None, None)
assert JSON.ToJSON(store).Reachable(Node, back) == text_
assert YAML.ToYAML(store).Reachable(Node, YAML.FromYAML(B).Reachable(Node, YAML.ToYAML(store).Reachable(Node, a))) == YAML.ToYAML(store).Reachable(Node, a)
assert Validators.Validate(B).Reachable(Node, back) == []
assert C.OfObject(Node, a).compare(C.OfObject(Node, back)) == 0

## BND-02 · A builder holds a state: natives checked by type, fixed and exclusive properties, other values as plain data

In [ ]:
a = Point("a", 5, {"w": 3}, ["p", "q"])
builder = Bd.Builder(BINDING, a)
names = []
builder.properties(lambda p: names.append(p.name()))
assert names == ["tag", "label", "i", "shape", "tags"]
assert builder.has("tag") and builder.has("i") and not builder.has("f") and not builder.has("nope")
assert "label" not in Bd.Builder(BINDING, Point()).state.values  # a builder drops absent values
# Natives: set, read back, type-checked, cleared.
builder.property("label", native("a2"))
got = []
builder.property("label", lambda p: p.value(lambda v: v.as_native(lambda n: got.append((n.has(), n.get())))))
assert got == [(True, "a2")]
with raises(TypeError, match="expected str, got int"):
    builder.property("label", native(1))
with raises(AttributeError, match="property 'f' is not set"):
    builder.property("f", lambda p: p.value(lambda v: v.as_native(lambda n: n.get())))
for method in ["as_object", "as_union", "as_intersection", "as_indexed"]:
    with raises(TypeError, match="property 'label' is native"):
        builder.property("label", lambda p, m=method: p.value(lambda v: getattr(v, m)(lambda x: None)))
# Exclusive: writing one clears the others of its group.
builder.property("f", native(2.5))
assert not builder.has("i") and builder.clone().number == 2.5
# Fixed: it always holds the binding's value, writing another raises, and clearing changes nothing.
builder.property("tag", native("node"))
with raises(ValueError, match="expected tag 'node', got 'other'"):
    builder.property("tag", native("other"))
with raises(TypeError, match="expected str, got int"):
    builder.property("tag", native(1))
builder.clear("tag").property("tag", lambda p: p.clear())
assert builder.has("tag")
# Other values, through their plain form: a value object, a list.
builder.property("shape", lambda p: p.value(lambda v: v.as_object(lambda o: o.property("w", native(9)))))
builder.property("tags", lambda p: p.value(lambda v: v.as_indexed(lambda l: l.append(lambda i: i.as_native(lambda n: n.set("r"))))))
made = builder.clone()
assert made.shape == {"w": 9} and made.tags == ["p", "q", "r"] and made.label == "a2"
builder.property("shape", lambda p: p.clear()).clear("tags").clear("nope").property("label", lambda p: p.clear())
made = builder.clone()
assert made.shape is None and made.tags is None and made.label is None
with raises(KeyError, match="unknown property 'nope'"):
    builder.property("nope", lambda p: None)

## BND-03 · A builder reads and adds entries; an implied adjacency keeps none

In [ ]:
b, c = Point("b"), Point("c")
builder = Bd.Builder(BINDING, Point("a", out=[(b, 7, {"text": "hi"})]))
names = []
builder.adjacencies(lambda adj: names.append((adj.name(), adj.me())))
assert names == [("out", "source"), ("into", "target")]
seen = []
builder.adjacency("out", lambda adj: adj.entries(lambda e: e.links(lambda k: (seen.append(k.name()), k.target(seen.append)))
                                                  .properties(lambda p: seen.append(p.name()))))
assert seen == ["target", b, "weight", "note"]


def fill(e):
    e.link("target", lambda k: k.set(c)).property("weight", native(1))
    assert e.has("weight") and not e.has("note") and not e.has("nope")
    e.clear("note")


builder.adjacency("out", lambda adj: adj.add(fill))
assert [(t.label, w) for t, w, _ in builder.clone().out] == [("b", 7), ("c", 1)]
props = []
builder.adjacency("out", lambda adj: adj.entries(lambda e: e.properties(lambda p: props.append(p.name()))))
assert props == ["weight", "note", "weight"]  # an absent property is not visited
for wrong, error, message in [
    (lambda e: e.link("source", lambda k: None), KeyError, "'source' is not a link this entry can set"),
    (lambda e: e.property("nope", lambda p: None), KeyError, "unknown property 'nope'"),
    (lambda e: e.link("target", lambda k: k.target(seen.append)), ValueError, "link 'target' is not set"),
]:
    with raises(error, match=message):
        builder.adjacency("out", lambda adj, w=wrong: adj.add(w))
builder.adjacency("out", lambda adj: adj.entries(lambda e: e.link("target", lambda k: k.target(
    lambda t: adj.remove(e) if t is b else None))).remove(None))  # removing what is not an entry changes nothing
assert [t.label for t, *_ in builder.clone().out] == ["c"]
# An implied adjacency: the other ends imply its entries, so those added are ignored, and none are read.
builder.adjacency("into", lambda adj: adj.add(lambda e: e.link("source", lambda k: k.set(c))).entries(seen.append).remove(None))
assert "into" not in builder.state.entries
with raises(KeyError, match="unknown adjacency 'nope'"):
    builder.adjacency("nope", lambda adj: None)
assert builder.identify(b) is builder

## BND-04 · Finalizing a builder, and the registry

In [ ]:
a = Point("a")
assert isinstance(Bd.Builder(BINDING).create(), Point) and Bd.Builder(BINDING, a).clone() is not a
updated = Bd.Builder(BINDING, a)
updated.property("label", native("a2"))
assert updated.update() is a and a.label == "a2"
with raises(ValueError, match="create() is only valid without a source instance; use clone() or update()"):
    Bd.Builder(BINDING, a).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    Bd.Builder(BINDING).clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    Bd.Builder(BINDING).update()
assert Bd.Builder(BINDING).adjacencies(lambda adj: None).state.entries == {"out": []}  # an implied one has none
assert Bd.Binding(Node, read, make).assign(a, Bd.State()) is a  # by default, an instance is not written back
# The registry: builders by schema name, as decoding needs them.
assert B.schema("Node") is Node and B.name_of(Node) == "Node" and B.name_of(Link) == "Link" and B.member(a, "label") == "a2"
assert isinstance(B.Node(), Bd.Builder) and isinstance(getattr(B, "Node")(a), Bd.Builder)
with raises(TypeError, match="'Link' is a relation; no relation builder is exposed"):
    B.schema("Link")
with raises(AttributeError, match="no schema registered as 'nope'"):
    B.schema("nope")
with raises(LookupError, match="schema is not registered"):
    B.name_of(Shape)
for name in ["nope", "_private"]:
    with raises(AttributeError):
        getattr(B, name)
# A fixed value is compared as a value, bytes too; a registry may know no relations.
Blob = S.OfObject.Builder().ref().properties(text("b", bytes)).create()
blobs = Bd.OfStore({"Blob": (Blob, lambda instance=None: Bd.Builder(Bd.Binding(Blob, read, make, fixed={"b": b"\x01"})))})
blobs.Blob().property("b", native(bytes([1])))
with raises(ValueError, match="expected b b'\\x01', got b'\\x02'"):
    blobs.Blob().property("b", native(b"\x02"))
assert blobs.name_of(Blob) == "Blob"
with raises(ValueError, match="got b'\\x01\\x02'"):
    blobs.Blob().property("b", native(b"\x01\x02"))